# Tool-Calling Research Agent (Teacher)

**Phase 1** of the BBB conference demo pipeline.

This notebook implements a tool-calling agent using the **OpenAI Responses API**
with reasoning support (gpt-5.4). The agent researches a given stock ticker by
calling financial tools (yfinance), then produces a structured research memo.

The full conversation trajectory (including reasoning summaries and all tool calls)
is saved as training data for fine-tuning Qwen3-4B.

## Architecture
```
User prompt → GPT-5.4 (Responses API)
                ├── reasoning item (summary captured)
                └── function_call item → execute tool → function_call_output → loop
                                                                                 ↓
                                                                         final message
```

## Setup

In [1]:
import json
import os
import sys
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv

# Add nb/ to path so we can import bbb as a package
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "nb"))

from bbb.tools import TOOL_SCHEMAS, TOOL_FUNCTIONS
from bbb.agent import run_tool_calling_agent

load_dotenv(PROJECT_ROOT / ".env")

client = OpenAI(base_url="https://us.api.openai.com/v1")

MODEL = "gpt-5.4"

## System Prompt

Uses the `developer` role (Responses API equivalent of `system`).
Instructs the model to use tools for research and produce a structured memo.
Reasoning happens via the model's internal reasoning tokens — no fake `<think>` tags needed.

In [2]:
SYSTEM_PROMPT = """\
You are an equity research analyst conducting comprehensive research on a given stock.

## Instructions
- Use the available tools to gather financial data, news, price history, and analyst recommendations.
- Before each tool call, take time to think about the task at hand.
- After gathering sufficient data, synthesize your findings into a structured research memo.
- Be thorough but efficient — aim for just the necessary amount of tool calls.

## Output Format
Your final response should be a markdown research memo with these sections:
- **Company Overview**: Brief description and market position
- **Financial Analysis**: Revenue, profitability, balance sheet highlights
- **Market Performance**: Price trends, volatility, trading patterns
- **Analyst Sentiment**: Consensus recommendations, recent upgrades/downgrades
- **Key Risks & Opportunities**: Summary of bull/bear case

## Important
- Use ONLY the tools provided. Do not make up financial data.
- Present facts objectively. Do not give investment advice.
- If a tool returns an error, note it and move on.
"""

## Tool Schemas

These are defined in `tools/stock_tools.py` and shared across all BBB notebooks.

In [3]:
# Quick look at what tools the agent has access to
# Responses API uses flat format: name/description/parameters at top level
for tool in TOOL_SCHEMAS:
    params = list(tool["parameters"]["properties"].keys())
    print(f"  {tool['name']}({', '.join(params)})")
    print(f"    → {tool['description']}\n")

  get_stock_news(ticker)
    → Get the 5 most recent news articles for a stock ticker.

  get_financials(ticker, statement_type, period)
    → Get financial statements (income, balance_sheet, or cashflow) for a stock.

  get_price_history(ticker, period, interval)
    → Get historical stock price data with summary statistics.

  get_recommendations(ticker, months_back)
    → Get analyst recommendations and recent upgrades/downgrades for a stock.



## Run the Agent

Uses `client.responses.create()` with reasoning enabled.
The agent loop handles:
- Parsing `function_call` items from `response.output`
- Executing tools and sending `function_call_output` back
- Passing reasoning items between turns (required for reasoning models)
- Capturing reasoning summaries for inspection

In [4]:
result = run_tool_calling_agent(
    client=client,
    model=MODEL,
    user_prompt="Research NVDA focusing on financial health and growth potential.",
    system_prompt=SYSTEM_PROMPT,
    reasoning_effort="medium",
)

print(f"\nTotal input items: {len(result['input'])}")
print(f"Reasoning summaries: {len(result['reasoning_summaries'])}")
print(f"Tokens — input: {result['usage']['input_tokens']}, output: {result['usage']['output_tokens']}, reasoning: {result['usage']['reasoning_tokens']}")

  [1] Reasoning: **Planning NVDA Research**

I need to research NVDA using various tools. It's important to think through each tool call ...
  [1] Called get_financials(ticker='NVDA', statement_type='income', period='annual')
  [1] Called get_financials(ticker='NVDA', statement_type='balance_sheet', period='annual')
  [1] Called get_financials(ticker='NVDA', statement_type='cashflow', period='annual')
  [1] Called get_financials(ticker='NVDA', statement_type='income', period='quarterly')
  [1] Called get_price_history(ticker='NVDA', period='1y', interval='1wk')
  [1] Called get_recommendations(ticker='NVDA', months_back=12)
  [1] Called get_stock_news(ticker='NVDA')
  [2] Reasoning: **Calculating financial metrics**

I need to compute margins and growth rates from the annual income figures. With reven...
  [2] Reasoning: **Analyzing market performance**

Operating income has shown growth alongside market performance. Over the past year, th...
  [2] Reasoning: **Researching tech news up

### Inspect the full trajectory

Everything in order: developer prompt → user → reasoning → tool calls → tool results → ... → final memo.

In [5]:
# Print the full trajectory: input items + final output items, all in order
all_items = list(result["input"]) + list(result["output"])

for i, item in enumerate(all_items):
    # Dict items (our input messages and function_call_outputs)
    if isinstance(item, dict):
        role = item.get("role", item.get("type", "?"))
        content = item.get("content", item.get("output", ""))
        if role == "developer":
            print(f"[{i}] DEVELOPER\n{str(content)[:200]}...\n")
        elif role == "user":
            print(f"[{i}] USER\n{content}\n")
        elif item.get("type") == "function_call_output":
            print(f"[{i}] TOOL RESULT (call_id: {item.get('call_id', '?')[:20]})\n{str(content)[:200]}...\n")
        else:
            print(f"[{i}] {role.upper()}\n{str(content)[:200]}...\n")
    # SDK response objects
    else:
        item_type = getattr(item, "type", "unknown")
        if item_type == "reasoning":
            summary = "(no summary)"
            if getattr(item, "summary", None) and item.summary:
                summary = item.summary[0].text
            print(f"[{i}] REASONING\n{summary}\n")
        elif item_type == "function_call":
            print(f"[{i}] FUNCTION CALL: {item.name}\n  args: {item.arguments}\n")
        elif item_type == "message":
            text = ""
            if hasattr(item, "content") and item.content:
                text = item.content[0].text
            print(f"[{i}] ASSISTANT (final)\n{text[:500]}...\n")
        else:
            print(f"[{i}] {item_type.upper()}\n{str(item)[:200]}\n")

[0] DEVELOPER
You are an equity research analyst conducting comprehensive research on a given stock.

## Instructions
- Use the available tools to gather financial data, news, price history, and analyst recommendat...

[1] USER
Research NVDA focusing on financial health and growth potential.

[2] REASONING
**Planning NVDA Research**

I need to research NVDA using various tools. It's important to think through each tool call carefully. I’m likely going to use parallel calls for different aspects like financials, prices, recommendations, and news. I want to gather information on annual income, balance sheets, and cash flow to assess financial health and growth potential, considering quarterly data too. 

Checking the market performance over 1 to 2 years, along with recent news, is also essential. I'll aim for efficiency by using seven tools in parallel.

[3] ASSISTANT (final)
I’ll pull the latest fundamentals, recent operating trend, price action, analyst sentiment, and recent news in pa

In [6]:
from IPython.display import Markdown, display

if result["output_text"]:
    display(Markdown(result["output_text"]))
else:
    print("No final text output — agent may have hit max iterations")

# NVDA Research Memo

## **Company Overview**
NVIDIA is a leading semiconductor and computing platform company, best known for GPUs and AI accelerators used across data centers, gaming, professional visualization, automotive, and software ecosystems. Its market position is strongest in AI infrastructure, where it benefits from a combination of hardware leadership, software lock-in, and broad ecosystem adoption.

From a growth-potential standpoint, NVIDIA appears to be operating from a position of financial strength rather than balance-sheet stress: it is highly profitable, strongly cash generative, and carries modest debt relative to liquidity.

---

## **Financial Analysis**

### Revenue and earnings growth
NVIDIA’s recent financial performance shows exceptional scale-up:

| Fiscal year ended Jan. 31 | Revenue | YoY Growth | Net Income | YoY Growth |
|---|---:|---:|---:|---:|
| 2024 | $60.9B | — | $29.8B | — |
| 2025 | $130.5B | +114.2% | $72.9B | +144.9% |
| 2026 | $215.9B | +65.5% | $120.1B | +64.7% |

Key takeaways:
- Revenue has more than tripled in two years.
- Net income expanded even faster than revenue over that span.
- Growth remains large even off a much bigger base, which supports the view that demand has not yet normalized sharply.

### Profitability
Margins remain extremely high:

- **FY2026 gross margin:** ~**71.1%**
- **FY2026 operating margin:** ~**60.4%**
- **FY2026 net margin:** ~**55.6%**

Compared with FY2025:
- Gross margin eased from roughly **75.0%** to **71.1%**
- Operating margin eased from roughly **62.4%** to **60.4%**
- Net margin stayed broadly stable near the mid-50s

Interpretation:
- Profitability remains elite.
- There is some evidence of normalization in gross/operating margin as revenue mix and infrastructure costs evolve, but not enough to change the overall picture of very strong earnings power.

### Recent quarterly trend
Quarterly revenue continued to rise sequentially:

| Quarter | Revenue | Net Income | Diluted EPS |
|---|---:|---:|---:|
| 2025-01-31 | $39.3B | $22.1B | $0.89 |
| 2025-04-30 | $44.1B | $18.8B | $0.76 |
| 2025-07-31 | $46.7B | $26.4B | $1.08 |
| 2025-10-31 | $57.0B | $31.9B | $1.30 |
| 2026-01-31 | $68.1B | $43.0B | $1.76 |

Latest-quarter observations:
- Revenue grew about **73% YoY** versus the comparable prior-year quarter.
- Net income grew about **95% YoY**.
- Revenue also rose about **19.5% sequentially** from the prior quarter.

That combination suggests NVIDIA still has meaningful near-term growth momentum, not just a high level of sales.

### Cash flow and capital allocation
Cash generation is a major strength:

| Fiscal year ended Jan. 31 | Operating Cash Flow | Free Cash Flow |
|---|---:|---:|
| 2024 | $28.1B | $27.0B |
| 2025 | $64.1B | $60.9B |
| 2026 | $102.7B | $96.7B |

Highlights:
- **FY2026 free cash flow:** **$96.7B**
- **FY2026 FCF margin:** ~**44.8%**
- Capex rose to **$6.0B**, but remained modest relative to operating cash generation.

Capital allocation:
- NVIDIA repurchased **$40.1B** of stock in FY2026.
- Dividends remained small at **$974M**.
- It also spent **$14.5B** on business acquisitions in FY2026, which contributed to higher goodwill/intangibles.

### Balance sheet health
NVIDIA’s balance sheet looks very strong:

- **Cash, cash equivalents, and short-term investments:** **$62.6B**
- **Total debt:** **$11.0B**
- **Stockholders’ equity:** **$157.3B**
- **Working capital:** **$93.4B**
- **Current assets:** **$125.6B**
- **Current liabilities:** **$32.2B**

Implications:
- Liquidity is ample.
- Debt appears manageable relative to cash resources and earnings power.
- The company has substantial flexibility for buybacks, acquisitions, and internal investment.

Areas to watch:
- **Accounts receivable** rose to **$38.5B** from **$23.1B** the prior year.
- **Inventory** increased to **$21.4B** from **$10.1B**.
- Those increases may reflect rapid scaling and customer demand, but they also raise execution risk if AI spending slows or product mix shifts.

---

## **Market Performance**
Over the last year, NVDA shares have been strong but volatile.

### Price summary
- **1-year total change:** **+56.2%**
- **1-year low:** **$94.29**
- **1-year high:** **$202.47**
- **Average price over period:** **$165.76**
- **Latest weekly close in dataset:** **$171.24**

### Trading pattern
- The stock fell sharply in early April 2025 to the period low near **$94**.
- It then staged a strong rally through summer and autumn, peaking above **$200** in late October 2025.
- Since that high, shares have pulled back, with the latest close about **15% below** the 1-year high.

### Volatility
- Weekly ranges and trading volumes were large, especially during the spring selloff and recovery.
- This pattern suggests the stock remains highly sensitive to:
  - AI demand expectations
  - valuation compression/expansion
  - broader megacap tech sentiment

For a company with NVIDIA’s growth profile, the market has rewarded fundamentals but also imposed meaningful swings in sentiment.

---

## **Analyst Sentiment**
Analyst sentiment is overwhelmingly positive.

### Current consensus snapshot
Latest recommendation mix:
- **Strong Buy:** 9
- **Buy:** 48
- **Hold:** 2
- **Sell:** 1
- **Strong Sell:** 0

That implies:
- **57 of 60 ratings** are Buy/Strong Buy
- Only **2 Holds** and **1 Sell**

### Recent rating actions
Recent actions were mostly reaffirmations or target increases:
- **Raymond James:** Strong Buy, target raised to **$323**
- **Truist:** Buy, target raised to **$287**
- **Wedbush:** Outperform, target raised to **$300**
- **JP Morgan:** Overweight, target raised to **$265**
- **Citigroup:** Buy, target raised to **$300**
- **Morgan Stanley:** Overweight, target raised to **$260**
- **BofA:** Buy, target raised to **$300**

Notable exception:
- **Seaport Global** initiated **Sell** in April 2025 with a **$100** target.

Overall interpretation:
- The Street remains highly constructive on NVIDIA’s growth outlook.
- Consensus appears to assume continued AI infrastructure demand and sustained earnings power.
- The dispersion in targets still indicates debate around valuation and durability, not broad concern about solvency or near-term financial health.

---

## **Key Risks & Opportunities**

### Opportunities / bull case
- **Sustained AI demand:** Revenue and earnings growth remain extraordinary even at a much larger scale.
- **Very strong financial health:** Large net cash position, high margins, and massive free cash flow support resilience.
- **Operating leverage:** R&D is rising, but revenue growth is outpacing opex growth.
- **Capital flexibility:** NVIDIA can continue buybacks, acquisitions, and ecosystem investment without balance-sheet strain.
- **Ecosystem advantage:** Recent news flow still points to NVIDIA being embedded across enterprise AI partnerships and infrastructure rollouts.

### Risks / bear case
- **Growth deceleration risk:** Expectations are extremely high; even strong growth could disappoint if it moderates faster than forecast.
- **Working capital buildup:** Sharp increases in receivables and inventory deserve monitoring for signs of demand digestion or shipment timing issues.
- **Margin normalization:** Gross and operating margins, while still exceptional, were modestly lower year over year.
- **Valuation/sentiment volatility:** The stock’s 1-year trading pattern shows it is vulnerable to sharp corrections even without a major fundamental breakdown.
- **Concentration and competition:** Large AI infrastructure customers and rival chip platforms remain ongoing strategic risks.

---

## Bottom line
NVIDIA’s **financial health is excellent**: high liquidity, modest leverage, exceptional profitability, and enormous free cash flow. Its **growth potential remains strong**, supported by continued quarterly revenue acceleration and broad analyst confidence. The main debate is less about balance-sheet quality and more about how long this level of AI-driven growth and margin strength can persist relative to already-elevated expectations.

## Save the Trajectory

We serialize the input list for training data. Responses API items
that are SDK objects get serialized via their `.model_dump()` method.

In [7]:
def serialize_input_list(input_list: list) -> list[dict]:
    """Convert a mix of dicts and SDK response objects to JSON-serializable dicts."""
    serialized = []
    for item in input_list:
        if isinstance(item, dict):
            serialized.append(item)
        elif hasattr(item, "model_dump"):
            serialized.append(item.model_dump())
        else:
            serialized.append({"type": "unknown", "data": str(item)})
    return serialized


output_dir = PROJECT_ROOT / "data" / "bbb"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "tool_calling_trajectories.jsonl"

record = {
    "input": serialize_input_list(result["input"]),
    "output": serialize_input_list(result["output"]),
    "tools": TOOL_SCHEMAS,
    "reasoning_summaries": result["reasoning_summaries"],
    "usage": result["usage"],
}

with open(output_file, "a") as f:
    f.write(json.dumps(record) + "\n")

print(f"Saved trajectory to {output_file}")

Saved trajectory to /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/tool_calling_trajectories.jsonl


## Test with a Few More Tickers

In [ ]:
test_tickers = [
    ("AAPL", "competitive position and market share"),
    ("TSLA", "recent news and analyst sentiment"),
    ("JPM", "financial health and balance sheet strength"),
]

for ticker, focus in test_tickers:
    print(f"\n{'='*60}")
    print(f"Researching {ticker} — {focus}")
    print(f"{'='*60}")
    
    res = run_tool_calling_agent(
        client=client,
        model=MODEL,
        user_prompt=f"Research {ticker} focusing on {focus}.",
        system_prompt=SYSTEM_PROMPT,
        reasoning_effort="medium",
    )
    
    # Count tool calls from the input list
    n_tool_results = sum(
        1 for item in res["input"]
        if isinstance(item, dict) and item.get("type") == "function_call_output"
    )
    memo_len = len(res["output_text"]) if res["output_text"] else 0
    print(f"  -> {n_tool_results} tool calls, {len(res['reasoning_summaries'])} reasoning steps, memo: {memo_len} chars")
    print(f"  -> Tokens: {res['usage']['input_tokens']} in / {res['usage']['output_tokens']} out / {res['usage']['reasoning_tokens']} reasoning")
    
    # Save
    record = {
        "input": serialize_input_list(res["input"]),
        "output": serialize_input_list(res["output"]),
        "tools": TOOL_SCHEMAS,
        "reasoning_summaries": res["reasoning_summaries"],
        "usage": res["usage"],
    }
    with open(output_file, "a") as f:
        f.write(json.dumps(record) + "\n")